# 01 · Supervised fine-tuning: chat templates, loss masking and packing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/post-training/01_sft_loss_masking.ipynb) &nbsp; 📖 Concept deck: [Supervised fine-tuning](https://nilesh-salpe.github.io/visual-ai/post-training/02-supervised-fine-tuning/) &nbsp; ⏱️ Suggested time: **35 min**

**You will build:** the data side of SFT: render a conversation with a chat template, train only on the assistant's tokens, compute the masked loss, and pack many short examples into fixed-length rows.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def log_softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    return x - np.log(np.exp(x).sum(axis=axis, keepdims=True))

def log_sigmoid(z):
    return -np.logaddexp(0, -z)

SPECIAL = ['<pad>', '<bos>', '<system>', '<user>', '<assistant>', '<eot>']
VOCAB = {t: i for i, t in enumerate(SPECIAL)}
def encode(text):
    """Toy word-level tokenizer: every new word gets the next id."""
    return [VOCAB.setdefault(w, len(VOCAB)) for w in text.split()]
PAD, BOS, EOT = VOCAB['<pad>'], VOCAB['<bos>'], VOCAB['<eot>']
IGNORE = -100   # the label value PyTorch's cross_entropy ignores by default

chat = [{'role': 'system', 'content': 'you are helpful'},
        {'role': 'user', 'content': 'what is two plus two'},
        {'role': 'assistant', 'content': 'two plus two is four'},
        {'role': 'user', 'content': 'and three plus three'},
        {'role': 'assistant', 'content': 'six'}]

def _render_ref(messages):
    tokens, mask = [BOS], [False]
    for m in messages:
        body = encode(m['content']); learn = m['role'] == 'assistant'
        tokens += [VOCAB['<' + m['role'] + '>']] + body + [EOT]; mask += [False] + [learn] * len(body) + [learn]
    return tokens, mask

## Task 1: Render a chat template with a loss mask

A chat model sees one flat token sequence. A **chat template** turns a list of messages into that sequence using special tokens. Ours (illustrative, Llama-style) is:
```
<bos> <role> content... <eot> <role> content... <eot> ...
```
Return `tokens` (list of ids) and `mask` (list of bools, same length). `mask[i]` is True when token *i* should be **learned**: the assistant's content tokens **and** its closing `<eot>`, so the model learns when to stop. Everything else is context: `<bos>`, role tokens, and system and user content.

Llama 2: *"we zero-out the loss on tokens from the user prompt, so as a result, we backpropagate only on answer tokens."*

In [ ]:
def render_chat(messages):
    """Returns (tokens, mask)."""
    # TODO: start with [BOS] / [False]; for each message add VOCAB['<'+role+'>'], encode(content), EOT
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_render_chat(f):
    toks, mask = f(chat)
    assert len(toks) == len(mask), "tokens and mask must have the same length"
    assert toks[0] == BOS and not mask[0], "first token is <bos> and is not trained"
    words = [k for t in toks for k, v in VOCAB.items() if v == t]
    assert words[:3] == ['<bos>', '<system>', 'you'], f"unexpected start: {words[:3]}"
    trained = [w for w, m in zip(words, mask) if m]
    assert trained == ['two', 'plus', 'two', 'is', 'four', '<eot>', 'six', '<eot>'], f"trained tokens should be the assistant answers + their <eot>, got {trained}"
    assert words.count('<eot>') == 5, "every message ends with <eot>"

check("render_chat", _test_render_chat, render_chat)

## Task 2: Shift and mask the labels

Next-token prediction: the input at position *t* predicts the token at *t+1*. So `inputs = tokens[:-1]` and `labels = tokens[1:]`. Replace every label whose **target token** is not trained with `IGNORE` (−100).

In [ ]:
def build_labels(tokens, mask):
    """Returns (inputs, labels) as numpy int arrays of length len(tokens) - 1."""
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_build_labels(f):
    toks, mask = _render_ref(chat)
    x, y = f(toks, mask)
    assert len(x) == len(y) == len(toks) - 1
    assert np.array_equal(x, toks[:-1]), "inputs are tokens[:-1]"
    keep = np.array(mask[1:])
    assert np.array_equal(y[keep], np.array(toks[1:])[keep]), "kept labels are the next tokens"
    assert np.all(y[~keep] == IGNORE), "untrained targets must be IGNORE"
    assert (y != IGNORE).sum() == 8

check("build_labels", _test_build_labels, build_labels)

## Task 3: The masked loss

Given `logits` [T, V] and `labels` [T], return the mean of $-\log p(\text{label}_t)$ over positions whose label is not `IGNORE`. The mean is over **trained tokens only**: averaging over all positions would quietly shrink the loss, and with it the effective learning rate.

In [ ]:
def masked_cross_entropy(logits, labels):
    # TODO: log_softmax, pick the label's log-prob where labels != IGNORE, average over those
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_masked_cross_entropy(f):
    r = np.random.default_rng(1); L = r.normal(size=(6, 10)); y = np.array([3, IGNORE, 1, IGNORE, IGNORE, 9])
    lp = log_softmax(L); want = -(lp[0, 3] + lp[2, 1] + lp[5, 9]) / 3
    assert np.isclose(f(L, y), want), "average -log p over non-ignored positions only"
    assert np.isclose(f(L, np.full(6, 4)), -lp[:, 4].mean())

check("masked_cross_entropy", _test_masked_cross_entropy, masked_cross_entropy)

## Task 4: Pack examples into fixed-length rows

Real SFT sets have many short examples. Padding each one to the max length wastes compute, so **pack** them: fill each row of length `max_len` with whole examples **in order**, starting a new row when the next example doesn't fit, then pad the rest (`PAD` input, `IGNORE` label). Llama 2: *"we concatenate all the prompts and answers from the training set"* to fill the sequence length.

Input: a list of `(inputs, labels)` pairs. Return `rows_x`, `rows_y` and `doc_ids` (int arrays [n_rows, max_len]; `doc_ids` = example index, −1 for padding). Pair this with the document mask from [Architecture notebook 05](https://github.com/nilesh-salpe/visual-ai/blob/main/build-it/architecture/05_attention_masks.ipynb) so examples can't attend to each other.

In [ ]:
def pack(examples, max_len):
    # TODO: greedily append each example to the current row if it fits, else start a new row
    raise NotImplementedError

In [ ]:
# @title Tests for Task 4: run after implementing
def _test_pack(f):
    r = np.random.default_rng(2)
    ex = [(np.arange(n), np.where(r.random(n) < .5, IGNORE, 1)) for n in [5, 3, 7, 2, 6, 1, 4]]
    X, Y, D = f(ex, 10)
    assert X.shape == Y.shape == D.shape and X.shape[1] == 10
    for i, (x, y) in enumerate(ex):
        sel = D == i
        assert sel.sum() == len(x), f"example {i} must appear exactly once, whole"
        assert np.array_equal(X[sel], x) and np.array_equal(Y[sel], y), f"example {i} content changed"
    assert np.all(Y[D == -1] == IGNORE) and np.all(X[D == -1] == PAD), "padding: PAD inputs, IGNORE labels"
    assert X.shape[0] == 4, f"greedy in-order packing should need 4 rows of 10 here, got {X.shape[0]}"

check("pack", _test_pack, pack)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Render a chat template with a loss mask
def render_chat_solution(messages):
    tokens, mask = [BOS], [False]
    for m in messages:
        learn = m['role'] == 'assistant'
        body = encode(m['content'])
        tokens += [VOCAB['<' + m['role'] + '>']] + body + [EOT]
        mask += [False] + [learn] * len(body) + [learn]
    return tokens, mask

check("render_chat (reference)", _test_render_chat, render_chat_solution)

In [ ]:
# @title Solution for Task 2: Shift and mask the labels
def build_labels_solution(tokens, mask):
    t = np.array(tokens)
    labels = np.where(np.array(mask[1:]), t[1:], IGNORE)
    return t[:-1], labels

check("build_labels (reference)", _test_build_labels, build_labels_solution)

In [ ]:
# @title Solution for Task 3: The masked loss
def masked_cross_entropy_solution(logits, labels):
    keep = labels != IGNORE
    lp = log_softmax(logits[keep])
    return -lp[np.arange(keep.sum()), labels[keep]].mean()

check("masked_cross_entropy (reference)", _test_masked_cross_entropy, masked_cross_entropy_solution)

In [ ]:
# @title Solution for Task 4: Pack examples into fixed-length rows
def pack_solution(examples, max_len):
    rows, cur = [], []
    for i, (x, y) in enumerate(examples):
        if sum(len(e[0]) for e in cur) + len(x) > max_len:
            rows.append(cur); cur = []
        cur.append((x, y, i))
    if cur: rows.append(cur)
    X = np.full((len(rows), max_len), PAD); Y = np.full((len(rows), max_len), IGNORE); D = np.full((len(rows), max_len), -1)
    for r_, row in enumerate(rows):
        p = 0
        for x, y, i in row:
            X[r_, p:p + len(x)] = x; Y[r_, p:p + len(x)] = y; D[r_, p:p + len(x)] = i; p += len(x)
    return X, Y, D

check("pack (reference)", _test_pack, pack_solution)

---
## Experiments
**How much of a chat is actually trained on?** In multi-turn data with long system prompts, the trained fraction can be small. That is fine, because the context still conditions the answer. Below: the trained fraction for our chat, and the padding waste of "one example per row" vs packing for random example lengths.

In [ ]:
toks, mask = render_chat_solution(chat)
print(f"chat: {len(toks)} tokens, {sum(mask)} trained ({sum(mask) / len(toks):.0%})")
r = np.random.default_rng(0); lens = r.integers(20, 400, 2000); L = 2048
ex = [(np.zeros(n, int), np.ones(n, int)) for n in lens]
X, Y, D = pack_solution(ex, L)
print(f"one example per row (padded to {lens.max()}): {lens.sum() / (len(lens) * lens.max()):.0%} of positions are real tokens")
print(f"packed into {X.shape[0]} rows of {L}: {(D >= 0).mean():.0%} real tokens")

---
## Takeaways
* SFT is ordinary next-token training on demonstrations. What makes it *instruction* tuning is the chat template and training only on the assistant's tokens.
* Train on the `<eot>` after each answer, or the model never learns to stop.
* Pack to avoid padding waste, with a document mask (or position resets) so examples don't leak into each other.
* Quality beats quantity: Llama 2 stopped at 27,540 high-quality SFT annotations ("Quality Is All You Need").